In [3]:
import ast
import json
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import geopandas as gpd
from tqdm import tqdm

datasets = Path("/nas/cee-ice/data")

save_dir = Path("/nas/cee-water/cjgleason/ted/swot-ml/data/multigraph_manual")
preprocess_dir = save_dir / "preprocess"
metadata_dir = save_dir / "metadata"

zarr_path = '/scratch4/workspace/tlanghorst_umass_edu-swot-ml-zarr'

matchups = gpd.read_parquet(metadata_dir / "gauges.parquet")#.set_index('comid')
# matchups.index = matchups.index.astype(str)

In [4]:
matchups

,name,area,active,latitude,longitude,last_updated,min_date,max_date,min_discharge,max_discharge,mean_discharge,count_discharge,provider_misc,geometry,provider,site_id,COMID,position,terminal_site_id
0,La rivière Bras David à Petit-Bourg [Prise d'e...,NaN,True,16.199137,-61.666436,2025-09-09,2017-03-09,2025-08-31,0.6180,267.999,5.229478,1121.0,None,POINT (-61.66615 16.19885),eauf,EAUF-10160001,76004387,0.913997,EAUF-10250001
1,La Grande Rivière à Goyaves à Sainte-Rose [La ...,NaN,True,16.279712,-61.668130,2025-09-09,1973-03-21,2025-08-31,0.7400,280.088,6.697796,4686.0,None,POINT (-61.66796 16.27954),eauf,EAUF-10250001,76004387,0.390817,EAUF-10250001
2,La rivière des Pères à Baillif,NaN,True,16.010760,-61.739270,2025-09-09,1983-04-29,2025-09-01,0.0380,37.724,2.024348,7643.0,None,POINT (-61.73927 16.01),eauf,EAUF-12120001,76004383,0.462555,EAUF-12120001
3,La Grande Rivière de Vieux-Habitants à Vieux-H...,NaN,True,16.086609,-61.725164,2025-09-09,1980-04-15,2025-08-31,0.3580,72.013,2.755552,10907.0,None,POINT (-61.73286 16.06881),eauf,EAUF-12210001,76004385,0.990000,EAUF-12210001
4,La rivière du Lorrain au Lorrain [SCNA],NaN,True,14.788162,-61.055505,2025-09-09,2011-02-09,2025-09-08,0.2980,34.989,2.505703,5061.0,None,POINT (-61.05534 14.788),eauf,EAUF-22050001,61000007,0.871666,EAUF-22050001
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
12111,PASSO DAS PEDRAS,6920.0,False,-32.519400,-53.455800,2025-11-06,2015-01-01,2025-08-31,1.5080,2789.540,129.902179,3799.0,None,POINT (-53.45833 -32.5194),brana,BRANA-88260000,64072596,0.329889,BRANA-88260000
12112,CERRO CHATO,1050.0,False,-31.864700,-53.268300,2025-11-06,1976-08-01,2025-06-30,0.3332,944.923,22.461594,16830.0,None,POINT (-53.26817 -31.86483),brana,BRANA-88575000,64068088,0.021428,BRANA-88641000
12113,PEDRO OSÓRIO,4700.0,False,-31.863300,-52.816100,2025-11-06,2000-04-19,2025-08-31,0.4346,7479.060,94.064795,9112.0,None,POINT (-52.81583 -31.86083),brana,BRANA-88641000,64067719,0.289136,BRANA-88641000
12114,PASSO DOS CARROS,131.0,False,-31.713900,-52.476700,2025-11-06,1964-10-15,2025-06-30,0.0028,176.979,3.001714,21711.0,None,POINT (-52.4764 -31.7136),brana,BRANA-88750000,64069237,0.088306,BRANA-88750000


In [2]:
from data import BasinDataLake

root_dir = save_dir / 'datalakes' / 'training'
store = BasinDataLake(root_dir)

In [13]:
import global_gauges as gg
facade = gg.GaugeDataFacade()

In [14]:
sites

,name,area,active,latitude,longitude,last_updated,min_date,max_date,min_discharge,max_discharge,mean_discharge,count_discharge,provider_misc,geometry,provider
site_id,,,,,,,,,,,,,,,
KRWAMIS-1001602,오대천평창군(송정교),229.84,True,37.624167,128.551111,2025-11-10,NaT,NaT,NaN,NaN,NaN,NaN,None,POINT (128.55111 37.62417),krwamis
KRWAMIS-1001603,N/A태백시(무사교),NaN,True,37.315000,128.994444,2025-11-10,2020-08-21,2025-11-10,0.1,67.0,1.720746,1340.0,None,POINT (128.99444 37.315),krwamis
KRWAMIS-1001605,송천송천1교,123.45,True,37.668056,128.709722,2025-11-10,NaT,NaT,NaN,NaN,NaN,NaN,None,POINT (128.70972 37.66806),krwamis
KRWAMIS-1001607,N/A삼척시(번천교),NaN,True,37.364444,128.996389,2025-11-10,NaT,NaT,NaN,NaN,NaN,NaN,None,POINT (128.99639 37.36444),krwamis
KRWAMIS-1001613,N/A삼척시(광동교),NaN,True,37.358611,128.948889,2025-11-10,NaT,NaT,NaN,NaN,NaN,NaN,None,POINT (128.94889 37.35861),krwamis
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
KRWAMIS-5302660,불갑천영광군(외간교),75.62,True,35.248889,126.466111,2025-11-10,NaT,NaT,NaN,NaN,NaN,NaN,None,POINT (126.46611 35.24889),krwamis
KRWAMIS-6001650,금성천제주시(개롱이교),64.60,True,33.436667,126.300833,2025-11-10,NaT,NaT,NaN,NaN,NaN,NaN,None,POINT (126.30083 33.43667),krwamis
KRWAMIS-6002670,한천제주시(제2동산교),33.14,True,33.495833,126.511944,2025-11-10,NaT,NaT,NaN,NaN,NaN,NaN,None,POINT (126.51194 33.49583),krwamis


In [21]:
sites_filt.to_file("/nas/cee-water/cjgleason/ted/gauge-match/data/krwamis.gpkg")

In [4]:
# processing_status = store.get_processing_status(source='gauge')
# processed_basins = processing_status.index.get_level_values('subbasin')
# to_process = matchups[~matchups.index.isin(processed_basins)]
to_process = matchups

max_batch_size = 64
for basin_id, basin_group in tqdm(to_process.groupby('outlet_id')):
    subbasin_data = {}
    for comid, row in basin_group.iterrows():
        if not row['custom']:
            gauge_df = None
        else:
            # There is not an explicit flag for custom that is not a gauge, so we just try
            # and then catch the error if this comid doesn't exist in our gauge dataset.
            try:
                gauge_df = facade.get_daily_values(comid).droplevel(axis=0, level=0)
                gauge_df = gauge_df[['discharge']]
                gauge_df['sp_discharge'] = gauge_df / row['total_area']
            except ValueError:
                gauge_df = None
                
        subbasin_data[comid] = gauge_df

        if len(subbasin_data)==max_batch_size:
            store.write_dynamic(basin_id, 'gauge', subbasin_data, mode='append')
            subbasin_data = {}
            
    # Write any remaining data.
    if len(subbasin_data)>0:
        store.write_dynamic(basin_id, 'gauge', subbasin_data, mode='append')

100%|██████████| 643/643 [3:00:58<00:00, 16.89s/it]    


In [ ]:
next(generate_tasks(matchups))

In [ ]:
gauge_df

In [ ]:
df = store.read_dynamic(basin_id, comid, 'gauge', '2020-01-01', '2020-12-31')['gauge']

In [ ]:
subbasin_data[comid]

In [ ]:
gauge_df['sp_discharge'].max()

In [ ]:
processed_basins = store.get_processing_status(source='swot-river')
processed_basins

In [ ]:
processed_basins[processed_basins['has_data']]['basin'].value_counts()

In [ ]:
store.compact()

In [ ]:
matchups['lake_pld_ids'].apply(len).gt(0).sum()